In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

In [31]:
DATA_DIR = Path(r"C:\Users\Jordi\Desktop\Trading\quant_lab\data\raw")
COLUMNS = ["date", "open", "high", "low", "close", "volume", "interest"]


resultats = []
for x in sorted(DATA_DIR.glob("*_RAD.CSV")):
    code = x.stem.removesuffix("_RAD")
    actif = pd.read_csv(x, names=COLUMNS)
    if actif.empty:
        continue
    actif["date"] = pd.to_datetime(actif["date"], format="%m/%d/%Y")

    rendement_max = actif['close'].pct_change().abs().max()

    first_date = actif["date"].min()
    last_date = actif["date"].max()
    jours_ouvres = len(pd.bdate_range(first_date, last_date))
    pct_missing = round(100 * (1 - len(actif) / jours_ouvres), 1)
    max_gap_days = actif["date"].diff().max().days

    if (pct_missing < 10) and (max_gap_days < 30) and (actif["close"].min() > 0) and (rendement_max < 0.5) and (actif["close"].notna().all()):

        resultats.append({
            "code": code,
            "rendement_max": rendement_max,
            "first_date": first_date,
            "last_date": last_date,
            "n_rows": len(actif),
            "max_gap_days": max_gap_days,
            "pct_missing": pct_missing,
            "min_close": actif["close"].min(),
        })
    else:
        print(f"Actif exclus {code}")

inventaire = pd.DataFrame(resultats).set_index("code")
inventaire

Actif exclus AD
Actif exclus AP
Actif exclus BG
Actif exclus BO
Actif exclus C_
Actif exclus ED
Actif exclus FA
Actif exclus FC
Actif exclus FX
Actif exclus GC
Actif exclus HG
Actif exclus HO
Actif exclus LC
Actif exclus LH
Actif exclus LX
Actif exclus NG
Actif exclus O_
Actif exclus PA
Actif exclus PL
Actif exclus S_
Actif exclus SF
Actif exclus SI
Actif exclus SM
Actif exclus TA
Actif exclus UA
Actif exclus US
Actif exclus W_
Actif exclus ZB
Actif exclus ZH
Actif exclus ZI
Actif exclus ZK
Actif exclus ZU


,rendement_max,first_date,last_date,n_rows,max_gap_days,pct_missing,min_close
code,,,,,,,
AN,0.094952,1988-01-04,2026-11-20,9807,4,3.3,32.420
AX,0.170548,1997-11-05,2026-11-20,7359,6,2.9,3104.500
BN,0.077836,1976-01-02,2026-11-20,12834,5,3.3,64.590
CA,0.126775,1999-01-04,2026-11-20,7122,6,2.1,1495.340
CB,0.022905,1994-06-01,2026-11-20,8094,6,4.5,47.720
...,...,...,...,...,...,...,...
ZR,0.102955,1987-01-02,2026-11-20,10037,5,3.5,1127.100
ZS,0.160768,1969-01-02,2026-11-20,14574,5,3.5,222.120
ZT,0.061596,1971-01-04,2026-11-20,14083,5,3.4,36.044


In [36]:
PROCESSED_DIR = DATA_DIR.parent / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

codes = inventaire.index
series = []
n_modifies = {}

for code in codes:
    actif = pd.read_csv(DATA_DIR / f"{code}_RAD.CSV", names=COLUMNS)
    actif["date"] = pd.to_datetime(actif["date"], format="%m/%d/%Y")
    close = actif.set_index("date")["close"]

    # Winsorisation : moyenne et écart-type exponentiels, demi-vie de 252 jours
    ewm = close.ewm(halflife=252)
    moyenne = ewm.mean()
    ecart_type = ewm.std()
    close_w = close.clip(lower=moyenne - 5 * ecart_type, upper=moyenne + 5 * ecart_type)

    n_modifies[code] = int((close_w != close).sum())
    rendement = close_w.pct_change()
    series.append(rendement.rename(code))

    # Un CSV par actif : prix winsorisé + rendement journalier
    pd.DataFrame({"close_w": close_w, "rendement": rendement}).to_csv(PROCESSED_DIR / f"{code}.csv")

rendements = pd.concat(series, axis=1, sort=True)
print(rendements.shape)
print(len(list(PROCESSED_DIR.glob("*.csv"))), "fichiers dans", PROCESSED_DIR)




(14827, 63)
63 fichiers dans C:\Users\Jordi\Desktop\Trading\quant_lab\data\processed
